# Task 1 population failure playground

Task 1 is easy to accidentally reduce to ‘predict the average expression vector at the next stage.’

That is not enough. The submission is a **population of cells**, and the scorer checks different properties of that population.

This notebook uses the organizers' tiny public E8.5/E9.5 examples and deliberately damages the known E9.5 target in controlled ways. Because we use the target to construct the examples, **this is a scorer/intuition experiment, not a model benchmark**.

We will make predictions that preserve one thing while breaking another:

- shuffle cell order, which should not matter
- repeat the target mean, which keeps pseudobulk nearly perfect but destroys population diversity
- shuffle each gene independently across cells, which keeps every per-gene marginal but destroys gene-gene structure
- resample only one cell state, which keeps real-looking cells but breaks the mixture

In [ ]:
# Pin the same public veckit commit for both code and sample data.
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" pandas matplotlib

In [ ]:
from pathlib import Path
import urllib.request

DATA = Path('/content/vec_t1_failures')
DATA.mkdir(exist_ok=True)
base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'

for name in ['sample_8.5.h5ad', 'sample_9.5.h5ad']:
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(base + name, p)
    print(name, round(p.stat().st_size / 1e6, 2), 'MB')

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from veckit import score

ref_path = DATA / 'sample_8.5.h5ad'
target_path = DATA / 'sample_9.5.h5ad'
target = ad.read_h5ad(target_path)

def dense_X(a):
    return a.X.toarray().astype(np.float32) if sparse.issparse(a.X) else np.asarray(a.X, dtype=np.float32)

X = dense_X(target)
print('target:', target.shape)
print('celltype labels available:', 'celltype' in target.obs)

## Build controlled failures

The point is to keep each intervention simple enough that we know exactly what changed.

**Row-order shuffle.** A set of cells has no meaningful ordering. Reordering rows should not change the score.

**Repeated mean.** Replace every cell by the same pseudobulk mean. The average expression is exactly right, but there is no cell-to-cell variation left.

**Gene-wise shuffle.** For each gene independently, permute its values among cells. Every gene keeps exactly the same empirical distribution, but the combinations of genes that co-occur inside a cell are broken.

**One-state resample.** Take cells from one training cell type and resample them until we have the original number of cells. These are real cells, but the population mixture is intentionally wrong.

In [ ]:
OUT = DATA / 'predictions'
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(0)

def save_pred(name, X_new=None, row_indices=None):
    if row_indices is not None:
        pred = target[row_indices].copy()
        pred.obs_names_make_unique()
    else:
        pred = target.copy()
        pred.X = np.asarray(X_new, dtype=np.float32)
    p = OUT / f'{name}.h5ad'
    pred.write_h5ad(p)
    return p

preds = {}
preds['row_order_only'] = save_pred('row_order_only', row_indices=rng.permutation(target.n_obs))
mean_cell = X.mean(axis=0, keepdims=True)
preds['repeated_mean'] = save_pred('repeated_mean', X_new=np.repeat(mean_cell, target.n_obs, axis=0))
X_gene_shuffle = X.copy()
for j in range(X.shape[1]):
    X_gene_shuffle[:, j] = X[rng.permutation(target.n_obs), j]
preds['gene_wise_shuffle'] = save_pred('gene_wise_shuffle', X_new=X_gene_shuffle)

if 'celltype' in target.obs:
    labels = target.obs['celltype'].astype(str).to_numpy()
    values, counts = np.unique(labels, return_counts=True)
    dominant = values[np.argmax(counts)]
    pool = np.flatnonzero(labels == dominant)
    idx = rng.choice(pool, size=target.n_obs, replace=True)
    preds['one_state_only'] = save_pred('one_state_only', row_indices=idx)
    print('one_state_only uses:', dominant, f'({len(pool)} source cells)')
else:
    print('No celltype labels in the mini target; skipping one_state_only.')
print('built:', list(preds))

## Verify what we preserved

Before looking at the official metrics, check the construction itself. The gene-wise shuffle should have essentially zero per-gene mean and variance error while still having a large covariance error.

In [ ]:
def quick_checks(path):
    A = ad.read_h5ad(path)
    Y = dense_X(A)
    pb_corr = np.corrcoef(Y.mean(0), X.mean(0))[0, 1]
    mean_err = np.mean(np.abs(Y.mean(0) - X.mean(0)))
    var_err = np.mean(np.abs(Y.var(0) - X.var(0)))
    keep = np.argsort(X.var(0))[-min(200, X.shape[1]):]
    cov_true = np.cov(X[:, keep], rowvar=False)
    cov_pred = np.cov(Y[:, keep], rowvar=False)
    cov_err = np.mean(np.abs(cov_pred - cov_true))
    return pb_corr, mean_err, var_err, cov_err

checks = pd.DataFrame({name: quick_checks(path) for name, path in preds.items()}, index=['pseudobulk_corr','mean_abs_error','variance_abs_error','covariance_abs_error']).T
display(checks.round(5))

## Run the official scorer

We score every controlled prediction against the known public E9.5 example, using E8.5 as the developmental reference. This is intentionally target-aware: the question is which metrics notice each kind of damage.

Useful fields are `de_score`, `de_direction`, `energy_distance`, `mmd_u`, `variogram`, `variance_ratio`, `composition_JSD`, and `pseudobulk_pearson`.

In [ ]:
metric_names = ['de_score','de_direction','energy_distance','mmd_u','variogram','variance_ratio','composition_JSD','pseudobulk_pearson']
rows = []
for name, path in preds.items():
    result = score(task='T1', input=path, target=target_path, reference=ref_path)
    m = result['metrics']
    rows.append({'prediction': name, **{k: m.get(k) for k in metric_names}})
scores = pd.DataFrame(rows).set_index('prediction')
display(scores.round(4))

## What should you learn?

The exact values are noisy because the bundled examples are tiny. The invariants matter more.

- **Row order should not matter.** The challenge does not assume cell `i` corresponds to cell `i` at another stage.
- **A perfect mean is not a population.** Repeating the mean can keep pseudobulk excellent while destroying variance, cell-state structure, and distributional metrics.
- **Correct one-gene-at-a-time histograms are not enough.** Gene-wise shuffling preserves every marginal but breaks which genes co-occur in the same cells.
- **Real-looking individual cells are not enough.** A population made from only one state can contain realistic cells and still have the wrong mixture.

For the bigger modeling picture, continue with the [ML guide](../ml-guide/guide.md).

Sources: https://virtualembryo.ai/challenge/evaluation?section=scoring&task=1 and https://github.com/aristoteleo/veckit